![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [6]:
import math
import numpy as np
import pandas as pd
from datetime import timedelta
from math import log, sqrt, exp
from scipy.stats import norm
from AlgorithmImports import * # QuantConnect specific imports

# --- Configuration ---

START_DATE = pd.Timestamp("2024-01-01")
# Reduce end date slightly to avoid issues with requesting data *on* the last day if needed for expiry calculations
END_DATE = pd.Timestamp("2024-12-30") 
UNIVERSE_RESOLUTION = Resolution.Daily
RISK_FREE_RATE = 0.02 # Annual risk-free rate assumption

# Define a broad equity universe (consider making this dynamic later)
TICKERS = [
    "COUP", "CSCO", "GILD", "FHN", "FIS", "T", "BMY", "ATVI", "BAC", "WFC", "F", "MANU", "CMCSA", "ORCL", "SHC",
    "NEE", "DAL", "DIS", "KO", "AAL", "OXY", "SCHW", "PYPL", "UAL", "MS", "SLB", "GM", "BX", "LVS", "VZ", "OSH", "RTX",
    "SPY", "TRU", "DELL"
    # Add back other tickers as needed and resources allow
]

# Pandas display settings (useful for notebooks)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 100) # Limit rows initially to avoid overwhelming output

In [7]:
# --- Black-Scholes Utility Functions ---

def bs_price(option_type: str, S: float, K: float, T: float, r: float, sigma: float) -> float:
    """
    Calculates the Black-Scholes theoretical value of a European call or put option.

    Args:
        option_type: 'C' for Call or 'P' for Put.
        S: Current price of the underlying asset.
        K: Strike price of the option.
        T: Time to maturity in years.
        r: Risk-free interest rate (annualized, decimal).
        sigma: Volatility of the underlying asset (annualized, decimal).

    Returns:
        Theoretical option price. Returns intrinsic value if T<=0 or sigma<=0.
    """
    if T <= 0 or sigma <= 0:
        # Return intrinsic value if time is up or volatility is zero/negative
        return max(0.0, (S - K) if option_type.upper() == 'C' else (K - S))

    try:
        d1 = (log(S / K) + (r + sigma**2 / 2) * T) / (sigma * sqrt(T))
        d2 = d1 - sigma * sqrt(T)
    except (ValueError, ZeroDivisionError): # Handle potential math errors (e.g., log(0), division by zero)
         return max(0.0, (S - K) if option_type.upper() == 'C' else (K - S))

    if option_type.upper() == 'C':
        price = S * norm.cdf(d1) - K * exp(-r * T) * norm.cdf(d2)
    elif option_type.upper() == 'P':
        price = K * exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    else:
        raise ValueError("option_type must be 'C' or 'P'")

    return max(0.0, price) # Price cannot be negative

def bs_vega(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """
    Calculates the Black-Scholes Vega (sensitivity to volatility).

    Args: See bs_price.

    Returns:
        Vega of the option. Returns 0 if T<=0 or sigma<=0.
    """
    if T <= 0 or sigma <= 0:
        return 0.0
    try:
        d1 = (log(S / K) + (r + sigma**2 / 2) * T) / (sigma * sqrt(T))
        vega = S * norm.pdf(d1) * sqrt(T)
    except (ValueError, ZeroDivisionError):
        return 0.0
    return vega

def bs_implied_vol(option_type: str, S: float, K: float, T: float, r: float, option_price: float, guess: float = 0.2, max_iter: int = 50, tol: float = 1.0e-5) -> float:
    """
    Calculates the Implied Volatility using the Newton-Raphson method.

    Args:
        option_type: 'C' or 'P'.
        S: Underlying price.
        K: Strike price.
        T: Time to maturity in years.
        r: Risk-free rate.
        option_price: Observed market price of the option.
        guess: Initial guess for volatility.
        max_iter: Maximum number of iterations.
        tol: Tolerance for convergence.

    Returns:
        Implied volatility (decimal). Returns NaN if unable to converge or invalid inputs.
    """
    if T <= 0 or option_price <= 0 or S <= 0 or K <= 0:
        # Return NaN for invalid inputs that preclude calculation
        return np.nan 

    # Check for arbitrage violations (minimum option price)
    min_price = max(0.0, (S - K * exp(-r * T)) if option_type.upper() == 'C' else (K * exp(-r * T) - S))
    if option_price < min_price - tol: # Check below intrinsic value bound
        return np.nan # Arbitrage, cannot calculate valid IV

    sigma = guess
    
    for i in range(max_iter):
        price = bs_price(option_type, S, K, T, r, sigma)
        vega = bs_vega(S, K, T, r, sigma)

        diff = price - option_price

        if abs(diff) < tol:
            return abs(sigma)

        # Avoid division by zero or very small vega
        if vega < 1.0e-8:
             # If vega is tiny, small price diffs can cause huge sigma jumps.
             # If price is already close, return current sigma. Otherwise, it might be far OTM/ITM.
             # Consider returning NaN or last sigma based on how close price is.
             return np.nan # Or perhaps abs(sigma) if abs(diff) is small enough relative to price

        sigma = sigma - diff / vega

        # Ensure sigma doesn't go non-positive or excessively large during iteration
        if sigma <= 0:
            sigma = tol # Reset to a small positive value if it goes non-positive
        elif sigma > 5.0: # Add an upper bound sanity check (e.g., 500%)
             sigma = 5.0

    # If loop finishes without converging
    # Check if the final price is reasonably close, otherwise return NaN
    final_price = bs_price(option_type, S, K, T, r, sigma)
    if abs(final_price - option_price) < tol * 10: # Looser tolerance for final check
        return abs(sigma)
    else:
        return np.nan # Failed to converge within tolerance

In [8]:
# --- QuantConnect Data Handling Functions ---

def setup_quantbook(start_date, end_date, benchmark="SPY") -> QuantBook:
    """Initializes and configures the QuantBook environment."""
    qb = QuantBook()
    qb.SetStartDate(start_date)
    # Note: QuantBook doesn't have SetEndDate, history requests use it.
    qb.SetBenchmark(benchmark)
    print(f"QuantBook initialized for {start_date.date()} to {end_date.date()}")
    return qb

def define_universe(qb: QuantBook, tickers: list, resolution: Resolution) -> list:
    """Adds equities and their corresponding options to the QuantBook subscription."""
    symbols = []
    print("Defining universe...")
    for ticker in tickers:
        try:
            equity = qb.AddEquity(ticker, resolution)
            # Check if equity symbol was added successfully (might fail for delisted etc.)
            if equity.Symbol not in qb.Securities:
                 print(f"Warning: Failed to add equity {ticker}")
                 continue
            option = qb.AddOption(equity.Symbol)
            symbols.append(equity.Symbol)
            # print(f" Added: {equity.Symbol} (Equity & Option)")
        except Exception as e:
            print(f"Error adding {ticker}: {e}")
    print(f"Universe defined with {len(symbols)} symbols.")
    return symbols

# --- QuantConnect Data Handling Functions ---

def fetch_and_merge_data(qb: QuantBook, symbols: list, start_date, end_date, resolution: Resolution) -> pd.DataFrame:
    """Fetches stock and option history, merges them for all symbols."""
    all_option_data = []
    print("Fetching and merging data...")
    
    # Create a list of specific symbols for the history request if needed, 
    # although requesting one by one is often clearer for debugging.
    
    for symbol in symbols:
        print(f" Processing: {symbol.Value}")
        try:
            # 1. Fetch Stock History for the specific symbol instance
            # CORRECTED: Pass the 'symbol' instance, not the type 'Symbol'
            # History for a single symbol usually returns a DataFrame directly
            stock_history_df = qb.History(symbol, start_date, end_date, resolution) 

            # Check if DataFrame is returned and not empty
            if stock_history_df is None or stock_history_df.empty:
                print(f"  No stock history for {symbol.Value}, skipping.")
                continue
                
            stock_history_df = stock_history_df.reset_index()
            
            # Check if 'time' and 'close' columns exist after reset_index
            if 'time' not in stock_history_df.columns or 'close' not in stock_history_df.columns:
                print(f"  Stock history for {symbol.Value} missing 'time' or 'close'. Columns: {stock_history_df.columns}. Skipping.")
                continue

            stock_history_df = stock_history_df[["time", "close"]] # Keep only necessary columns
            stock_history_df.rename(columns={"close": "underlying_price"}, inplace=True)
            
            # Ensure time column is suitable for merging (remove timezone if necessary)
            if stock_history_df['time'].dt.tz is not None:
                stock_history_df['time'] = stock_history_df['time'].dt.tz_localize(None)
            stock_history_df['time'] = stock_history_df['time'].dt.normalize() # Keep only date part for daily merge

            # 2. Fetch Option History
            # CORRECTED: Use .DataFrame property directly if available and suitable
            # Avoid GetSourceData().get(equity_symbol) which is likely incorrect pattern
            option_contracts_history = qb.OptionHistory(symbol, start_date, end_date, resolution)
            
            # Check if the history object is valid/contains data before accessing DataFrame
            if not option_contracts_history: # Or more specific check depending on return type
                print(f"  No option history object returned for {symbol.Value}, skipping.")
                continue

            option_history_df = option_contracts_history.DataFrame # Common way to get pandas DF

            if option_history_df is None or option_history_df.empty:
                print(f"  No option history DataFrame for {symbol.Value}, skipping.")
                continue
                
            # Make sure columns needed exist before reset_index
            required_cols = ['time', 'bidclose', 'askclose', 'strike', 'expiry', 'type']
            if not all(col in option_history_df.columns for col in required_cols):
                print(f" Missing required option columns for {symbol.Value}, skipping. Has: {option_history_df.columns}")
                continue
                
            option_history_df = option_history_df.reset_index()

            # 3. Merge Data
            # Ensure time column is suitable for merging
            if option_history_df['time'].dt.tz is not None:
                option_history_df['time'] = option_history_df['time'].dt.tz_localize(None)
            option_history_df['time'] = option_history_df['time'].dt.normalize() # Keep only date part

            # Add symbol column before merge if needed later
            # The option DataFrame usually includes the underlying symbol implicitly 
            # in its structure or index, but adding explicitly can be helpful.
            # Let's assume the 'symbol' column from reset_index refers to the *option* contract.
            # We need the *underlying* symbol for grouping/analysis later.
            option_history_df['underlying_symbol'] = symbol.Value 

            merged_df = option_history_df.merge(stock_history_df, on="time", how="inner") # Use inner to ensure we have both prices

            if merged_df.empty:
                print(f"  Merge resulted in empty DataFrame for {symbol.Value}, skipping.")
                continue

            all_option_data.append(merged_df)

        except KeyError as e:
            print(f"  KeyError processing {symbol.Value}: {e}. Check column names after history calls/reset_index. Skipping.")
        except AttributeError as e:
             print(f"  AttributeError processing {symbol.Value}: {e}. Often happens if history object is None or doesn't have expected property like .DataFrame. Skipping.")
        except Exception as e:
            # Catch other potential errors during history fetching/processing
            print(f"  Unexpected error processing {symbol.Value}: {type(e).__name__} - {e}. Skipping.")


    if not all_option_data:
        print("No data fetched for any symbol.")
        return pd.DataFrame() # Return empty DataFrame

    print("Concatenating data...")
    final_df = pd.concat(all_option_data, ignore_index=True)
    print(f"Finished fetching data. Total rows: {len(final_df)}")
    return final_df


In [9]:
# --- Data Preprocessing and Calculation Functions ---

def preprocess_options_data(df: pd.DataFrame) -> pd.DataFrame:
    """Performs basic preprocessing like calculating mid-price and time to expiry."""
    print("Preprocessing data...")
    if df.empty:
        print(" Input DataFrame is empty, skipping preprocessing.")
        return df
        
    # Ensure required columns exist
    required_cols = ['bidclose', 'askclose', 'expiry', 'time', 'underlying_price', 'strike']
    if not all(col in df.columns for col in required_cols):
        missing = [col for col in required_cols if col not in df.columns]
        print(f"Error: Missing required columns for preprocessing: {missing}")
        # Decide how to handle: raise error or return empty/original df
        return pd.DataFrame() # Return empty if crucial columns missing

    # Calculate Mid Price
    df["option_mid"] = (df["bidclose"] + df["askclose"]) / 2

    # Calculate Days to Expiry (ensure correct date handling)
    # Convert columns to datetime objects if they are not already
    df['expiry_dt'] = pd.to_datetime(df['expiry']).dt.normalize()
    df['time_dt'] = pd.to_datetime(df['time']).dt.normalize()
    df["days_to_expiry"] = (df['expiry_dt'] - df['time_dt']).dt.days

    # Calculate Time to Expiry in Years
    df["T_years"] = df["days_to_expiry"] / 365.0
    
    # Basic Cleaning - Remove rows where essential calculations might fail
    # e.g. non-positive prices, strike, or expiry already passed
    initial_rows = len(df)
    df.drop(df[df['days_to_expiry'] < 0].index, inplace=True) # Drop if already expired (shouldn't happen with daily data but good check)
    df.drop(df[df['option_mid'] <= 0].index, inplace=True) # Drop non-positive mid price
    df.drop(df[df['underlying_price'] <= 0].index, inplace=True) # Drop non-positive underlying price
    df.drop(df[df['strike'] <= 0].index, inplace=True) # Drop non-positive strike
    
    # Drop rows with NaN in critical input columns before IV calculation
    critical_iv_inputs = ['type', 'underlying_price', 'strike', 'T_years', 'option_mid']
    df.dropna(subset=critical_iv_inputs, how='any', inplace=True)

    cleaned_rows = len(df)
    print(f"Preprocessing finished. Rows before cleaning: {initial_rows}, after: {cleaned_rows}")
    
    # Remove temporary datetime columns if desired
    df.drop(columns=['expiry_dt', 'time_dt'], inplace=True)

    return df

def calculate_implied_volatility(df: pd.DataFrame, r: float) -> pd.DataFrame:
    """Calculates implied volatility for each option row."""
    print(f"Calculating Implied Volatility for {len(df)} rows...")
    if df.empty:
        print(" Input DataFrame is empty, skipping IV calculation.")
        return df
        
    # Define the helper function for apply
    def _compute_iv_row(row, risk_free_rate):
        opt_type = 'C' if (int(row["type"]) == 0) else 'P'
        S = row["underlying_price"]
        K = row["strike"]
        T = row["T_years"]
        px = row["option_mid"]
        # Pass valid inputs to the BS IV function
        return bs_implied_vol(opt_type, S, K, T, risk_free_rate, px)

    # Apply the function row-wise
    # !! Performance Warning: .apply() is slow on large DataFrames !!
    # For production/large scale, consider vectorization libraries (e.g., py_vollib)
    # or parallel processing if environment allows.
    df["implied_vol"] = df.apply(_compute_iv_row, axis=1, risk_free_rate=r)

    # Analyze IV results - how many NaNs?
    nan_iv_count = df['implied_vol'].isna().sum()
    print(f"Finished IV calculation. {nan_iv_count} rows resulted in NaN IV ({nan_iv_count / len(df):.2%}).")
    
    # Optionally drop rows where IV calculation failed
    # df.dropna(subset=['implied_vol'], inplace=True)
    # print(f"Rows after dropping NaN IVs: {len(df)}")

    return df


In [10]:
 # 1. Setup Environment
qb = setup_quantbook(START_DATE, END_DATE)

# 2. Define Universe
symbols = define_universe(qb, TICKERS, UNIVERSE_RESOLUTION)

# Check if any symbols were successfully added
if not symbols:
    print("No symbols were added to the universe. Exiting.")
else:
    # 3. Fetch and Merge Data (using the corrected function)
    raw_data_df = fetch_and_merge_data(qb, symbols, START_DATE, END_DATE, UNIVERSE_RESOLUTION)

    if raw_data_df.empty:
            print("No data was fetched or merged successfully. Exiting.")
    else:
        # 4. Preprocess Data (Calculate Mid, TTE etc.)
        preprocessed_df = preprocess_options_data(raw_data_df)

        if preprocessed_df.empty:
            print("Preprocessing resulted in an empty DataFrame. Exiting.")
        else:
            # 5. Calculate Implied Volatility
            final_options_df = calculate_implied_volatility(preprocessed_df, RISK_FREE_RATE)

            # Display some results
            print("\nSample of the final DataFrame:")
            print(final_options_df.head())
            print(f"\nTotal rows in final DataFrame: {len(final_options_df)}")

            print("\nData preparation complete. Ready for strategy implementation.")

In [11]:
import math
import numpy as np
import pandas as pd
from datetime import timedelta, datetime # Import datetime
from math import log, sqrt, exp
from scipy.stats import norm
from AlgorithmImports import * # QuantConnect specific imports

# --- Configuration ---

START_DATE = pd.Timestamp("2023-01-01") # Changed start date to 2023 for a larger history
# CORRECTED: Set END_DATE to a date in the past or use current time
# For reproducibility in research, a fixed past date is better.
# Let's use the end of 2023 as an example. Adjust as needed.
END_DATE = pd.Timestamp("2023-12-31")
# If you want data up to 'yesterday' dynamically (use with caution in notebooks):
# END_DATE = (datetime.utcnow() - timedelta(days=1)).date() 

UNIVERSE_RESOLUTION = Resolution.Daily
RISK_FREE_RATE = 0.02 # Annual risk-free rate assumption

# Define a broad equity universe (consider making this dynamic later)
TICKERS = [
    "COUP", "CSCO", "GILD", "FHN", "FIS", "T", "BMY", "ATVI", "BAC", "WFC", "F", "MANU", "CMCSA", "ORCL", "SHC",
    "NEE", "DAL", "DIS", "KO", "AAL", "OXY", "SCHW", "PYPL", "UAL", "MS", "SLB", "GM", "BX", "LVS", "VZ", "OSH", "RTX",
    "SPY", "TRU", "DELL"
    # Add back other tickers as needed and resources allow
]

# Pandas display settings (useful for notebooks)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 100) # Limit rows initially to avoid overwhelming output


# --- Black-Scholes Utility Functions ---
# (Keep the bs_price, bs_vega, bs_implied_vol functions as corrected in the previous response)
def bs_price(option_type: str, S: float, K: float, T: float, r: float, sigma: float) -> float:
    """
    Calculates the Black-Scholes theoretical value of a European call or put option.
    """
    if T <= 0 or sigma <= 0:
        return max(0.0, (S - K) if option_type.upper() == 'C' else (K - S))
    try:
        with np.errstate(divide='ignore', invalid='ignore'): # Suppress warnings for intermediate steps
            d1 = (log(S / K) + (r + sigma**2 / 2) * T) / (sigma * sqrt(T))
            d2 = d1 - sigma * sqrt(T)
            if not (np.isfinite(d1) and np.isfinite(d2)):
                 return max(0.0, (S - K) if option_type.upper() == 'C' else (K - S)) # intrinsic if d1/d2 invalid
    except (ValueError, ZeroDivisionError):
         return max(0.0, (S - K) if option_type.upper() == 'C' else (K - S))

    if option_type.upper() == 'C':
        price = S * norm.cdf(d1) - K * exp(-r * T) * norm.cdf(d2)
    elif option_type.upper() == 'P':
        price = K * exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    else:
        raise ValueError("option_type must be 'C' or 'P'")
    return max(0.0, price)

def bs_vega(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """
    Calculates the Black-Scholes Vega (sensitivity to volatility).
    """
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.0
    try:
        with np.errstate(divide='ignore', invalid='ignore'): # Suppress warnings
            d1 = (log(S / K) + (r + sigma**2 / 2) * T) / (sigma * sqrt(T))
            if not np.isfinite(d1):
                return 0.0
        vega = S * norm.pdf(d1) * sqrt(T)
        if not np.isfinite(vega):
             return 0.0
    except (ValueError, ZeroDivisionError):
        return 0.0
    return vega

def bs_implied_vol(option_type: str, S: float, K: float, T: float, r: float, option_price: float, guess: float = 0.2, max_iter: int = 50, tol: float = 1.0e-5) -> float:
    """
    Calculates the Implied Volatility using the Newton-Raphson method.
    """
    if T <= 0 or option_price <= 0 or S <= 0 or K <= 0:
        return np.nan
    min_price = max(0.0, (S - K * exp(-r * T)) if option_type.upper() == 'C' else (K * exp(-r * T) - S))
    if option_price < min_price - tol:
        return np.nan

    sigma = guess
    for i in range(max_iter):
        price = bs_price(option_type, S, K, T, r, sigma)
        vega = bs_vega(S, K, T, r, sigma)
        diff = price - option_price
        if abs(diff) < tol:
            return abs(sigma)
        if vega < 1.0e-8:
             # If vega is tiny, IV is unstable. Return NaN or check if price match is extremely close.
             return np.nan 
        sigma = sigma - diff / vega
        if sigma <= 0:
            sigma = tol 
        elif sigma > 5.0: # Upper bound sanity check (500%)
            sigma = 5.0 # Clamp to upper bound
            
    # Final check after max_iter
    final_price = bs_price(option_type, S, K, T, r, sigma)
    if abs(final_price - option_price) < tol * 10: # Use a slightly looser tolerance
        return abs(sigma)
    else:
        return np.nan # Failed to converge


# --- QuantConnect Data Handling Functions ---

def setup_quantbook(start_date, end_date, benchmark="SPY") -> QuantBook:
    """Initializes and configures the QuantBook environment."""
    qb = QuantBook()
    qb.SetStartDate(start_date)
    qb.SetBenchmark(benchmark)
    print(f"QuantBook initialized for {start_date.date()} to {end_date.date()}")
    return qb

def define_universe(qb: QuantBook, tickers: list, resolution: Resolution) -> list:
    """Adds equities and their corresponding options to the QuantBook subscription."""
    symbols = []
    print("Defining universe...")
    for ticker in tickers:
        try:
            equity = qb.AddEquity(ticker, resolution)
            if equity.Symbol not in qb.Securities:
                 print(f"Warning: Failed to add equity {ticker}")
                 continue
            # Check if options are available for this equity in QC data
            if not equity.HasOptions:
                 print(f"Warning: Options not available for {ticker}, skipping option subscription.")
                 # Decide if you still want the equity symbol if no options? 
                 # For this strategy, probably not.
                 continue 
            option = qb.AddOption(equity.Symbol)
            symbols.append(equity.Symbol)
            # print(f" Added: {equity.Symbol} (Equity & Option)")
        except Exception as e:
            print(f"Error adding {ticker}: {e}")
    print(f"Universe defined with {len(symbols)} equity symbols having options.")
    return symbols

def fetch_and_merge_data(qb: QuantBook, symbols: list, start_date, end_date, resolution: Resolution) -> pd.DataFrame:
    """Fetches stock and option history, merges them for all symbols."""
    all_option_data = []
    print(f"Fetching and merging data from {start_date.date()} to {end_date.date()}...") # Added date range print

    for symbol in symbols:
        print(f" Processing: {symbol.Value}")
        try:
            # 1. Fetch Stock History
            # ADDED: Print arguments for debugging
            # print(f"  Requesting Stock History: {symbol}, {start_date}, {end_date}, {resolution}")
            stock_history_df = qb.History(symbol, start_date, end_date, resolution)

            # More robust check
            if stock_history_df is None or stock_history_df.empty or 'close' not in stock_history_df.columns:
                print(f"  No valid stock history for {symbol.Value}. Columns: {stock_history_df.columns if stock_history_df is not None else 'None'}. Skipping.")
                continue

            stock_history_df = stock_history_df.reset_index()

            if 'time' not in stock_history_df.columns or 'close' not in stock_history_df.columns:
                print(f"  Stock history columns missing after reset_index: {stock_history_df.columns}. Skipping.")
                continue

            stock_history_df = stock_history_df[["time", "close"]]
            stock_history_df.rename(columns={"close": "underlying_price"}, inplace=True)
            if stock_history_df['time'].dt.tz is not None:
                stock_history_df['time'] = stock_history_df['time'].dt.tz_localize(None)
            stock_history_df['time'] = stock_history_df['time'].dt.normalize()

            # 2. Fetch Option History
            # print(f"  Requesting Option History: {symbol}, {start_date}, {end_date}, {resolution}")
            option_contracts_history = qb.OptionHistory(symbol, start_date, end_date, resolution)
            
            if not option_contracts_history: 
                print(f"  No option history object returned for {symbol.Value}, skipping.")
                continue

            option_history_df = option_contracts_history.DataFrame 

            if option_history_df is None or option_history_df.empty:
                print(f"  No option history DataFrame for {symbol.Value}, skipping.")
                continue
                
            required_cols = ['time', 'bidclose', 'askclose', 'strike', 'expiry', 'type']
            if not all(col in option_history_df.columns for col in required_cols):
                print(f" Missing required option columns for {symbol.Value}, skipping. Has: {option_history_df.columns}")
                continue
                
            option_history_df = option_history_df.reset_index()

            # 3. Merge Data
            if option_history_df['time'].dt.tz is not None:
                option_history_df['time'] = option_history_df['time'].dt.tz_localize(None)
            option_history_df['time'] = option_history_df['time'].dt.normalize() 
            option_history_df['underlying_symbol'] = symbol.Value 

            # print(f"  Stock df times head: {stock_history_df['time'].head()}")
            # print(f"  Option df times head: {option_history_df['time'].head()}")
            
            merged_df = option_history_df.merge(stock_history_df, on="time", how="inner") 

            if merged_df.empty:
                print(f"  Merge resulted in empty DataFrame for {symbol.Value} (check time alignment). Skipping.")
                continue

            all_option_data.append(merged_df)

        except KeyError as e:
            print(f"  KeyError processing {symbol.Value}: {e}. Check column names. Skipping.")
        except AttributeError as e:
             print(f"  AttributeError processing {symbol.Value}: {e}. Skipping.")
        except Exception as e:
            print(f"  Unexpected error processing {symbol.Value}: {type(e).__name__} - {e}. Skipping.")


    if not all_option_data:
        print("No data fetched for any symbol.")
        return pd.DataFrame() 

    print("Concatenating data...")
    final_df = pd.concat(all_option_data, ignore_index=True)
    print(f"Finished fetching data. Total rows: {len(final_df)}")
    return final_df


# --- Data Preprocessing and Calculation Functions ---
# (Keep preprocess_options_data and calculate_implied_volatility as corrected previously)
def preprocess_options_data(df: pd.DataFrame) -> pd.DataFrame:
    """Performs basic preprocessing like calculating mid-price and time to expiry."""
    print("Preprocessing data...")
    if df.empty:
        print(" Input DataFrame is empty, skipping preprocessing.")
        return df
    required_cols = ['bidclose', 'askclose', 'expiry', 'time', 'underlying_price', 'strike']
    if not all(col in df.columns for col in required_cols):
        missing = [col for col in required_cols if col not in df.columns]
        print(f"Error: Missing required columns for preprocessing: {missing}")
        return pd.DataFrame()

    df["option_mid"] = (df["bidclose"] + df["askclose"]) / 2
    df['expiry_dt'] = pd.to_datetime(df['expiry']).dt.normalize()
    df['time_dt'] = pd.to_datetime(df['time']).dt.normalize()
    df["days_to_expiry"] = (df['expiry_dt'] - df['time_dt']).dt.days
    df["T_years"] = df["days_to_expiry"] / 365.0
    initial_rows = len(df)
    df.drop(df[df['days_to_expiry'] < 0].index, inplace=True) 
    df.drop(df[df['option_mid'] <= 0].index, inplace=True) 
    df.drop(df[df['underlying_price'] <= 0].index, inplace=True) 
    df.drop(df[df['strike'] <= 0].index, inplace=True) 
    critical_iv_inputs = ['type', 'underlying_price', 'strike', 'T_years', 'option_mid']
    df.dropna(subset=critical_iv_inputs, how='any', inplace=True)
    cleaned_rows = len(df)
    print(f"Preprocessing finished. Rows before cleaning: {initial_rows}, after: {cleaned_rows}")
    if 'expiry_dt' in df.columns: df.drop(columns=['expiry_dt'], inplace=True)
    if 'time_dt' in df.columns: df.drop(columns=['time_dt'], inplace=True)
    return df

def calculate_implied_volatility(df: pd.DataFrame, r: float) -> pd.DataFrame:
    """Calculates implied volatility for each option row."""
    print(f"Calculating Implied Volatility for {len(df)} rows...")
    if df.empty:
        print(" Input DataFrame is empty, skipping IV calculation.")
        return df
    def _compute_iv_row(row, risk_free_rate):
        opt_type = 'C' if (int(row["type"]) == 0) else 'P'
        S = row["underlying_price"]
        K = row["strike"]
        T = row["T_years"]
        px = row["option_mid"]
        return bs_implied_vol(opt_type, S, K, T, risk_free_rate, px)

    tqdm.pandas(desc="Calculating IV") # Add progress bar if tqdm is available
    df["implied_vol"] = df.progress_apply(_compute_iv_row, axis=1, risk_free_rate=r)
    # If tqdm not available:
    # df["implied_vol"] = df.apply(_compute_iv_row, axis=1, risk_free_rate=r) 

    nan_iv_count = df['implied_vol'].isna().sum()
    total_rows = len(df) if len(df) > 0 else 1
    print(f"Finished IV calculation. {nan_iv_count} rows resulted in NaN IV ({nan_iv_count / total_rows:.2%}).")
    return df

# --- Main Execution Block ---
from tqdm.notebook import tqdm # Import tqdm for progress bar

if __name__ == "__main__":
    # 1. Setup Environment
    qb = setup_quantbook(START_DATE, END_DATE)

    # 2. Define Universe
    symbols = define_universe(qb, TICKERS, UNIVERSE_RESOLUTION)

    if not symbols:
        print("No symbols were added to the universe. Exiting.")
    else:
        # 3. Fetch and Merge Data
        raw_data_df = fetch_and_merge_data(qb, symbols, START_DATE, END_DATE, UNIVERSE_RESOLUTION)

        if raw_data_df.empty:
             print("No data was fetched or merged successfully. Exiting.")
        else:
            # 4. Preprocess Data
            preprocessed_df = preprocess_options_data(raw_data_df)

            if preprocessed_df.empty:
                print("Preprocessing resulted in an empty DataFrame. Exiting.")
            else:
                # 5. Calculate Implied Volatility
                final_options_df = calculate_implied_volatility(preprocessed_df, RISK_FREE_RATE)

                print("\nSample of the final DataFrame:")
                print(final_options_df.head())
                print(f"\nTotal rows in final DataFrame: {len(final_options_df)}")
                print("\nData preparation complete. Ready for strategy implementation.")